In [ ]:
import pandas as pd
import numpy as np
import joblib
import requests
import holidays

In [ ]:
MODEL_PATH = "spot_price_forecast_model.pkl"

SPOT_HISTORY_PATH = "SPOT_2026.xlsx"

FORECAST_RUN_TIME = pd.Timestamp("2026-03-28 08:00:00")
FORECAST_DATE = pd.Timestamp("2026-03-29").date()

LAT = 52.2297
LON = 21.0122

In [ ]:
model_package = joblib.load(MODEL_PATH)

model = model_package["best_model"]
features = model_package["features"]

print("Model:", model_package["best_model_name"])
print("Liczba cech:", len(features))

In [ ]:
spot_hist = pd.read_excel(SPOT_HISTORY_PATH)

spot_hist["timestamp"] = pd.to_datetime(spot_hist["timestamp"])
spot_hist["price_spot"] = pd.to_numeric(spot_hist["price_spot"], errors="coerce")

spot_hist = (
    spot_hist
    .dropna(subset=["timestamp", "price_spot"])
    .drop_duplicates(subset=["timestamp"])
    .sort_values("timestamp")
    .reset_index(drop=True)
)

# W dniu 2 lutego znamy już ceny na cały 2 lutego
last_known_price_time = pd.Timestamp(f"{FORECAST_DATE - pd.Timedelta(days=1)} 23:00:00")

spot_hist = spot_hist[spot_hist["timestamp"] <= last_known_price_time].copy()

spot_hist

In [ ]:
import requests
import time

def get_pse_json(endpoint, params=None):
    url = f"https://api.raporty.pse.pl/api/{endpoint}"
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    
    data = response.json()
    
    if "value" in data:
        return pd.DataFrame(data["value"])
    
    return pd.DataFrame(data)

In [ ]:
def download_pse_pk5l_wp(start_date, end_date):
    url = "https://api.raporty.pse.pl/api/pk5l-wp"
    
    start_date = pd.to_datetime(start_date).strftime("%Y-%m-%d")
    end_date_plus_1 = (pd.to_datetime(end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    params = {
        "$filter": (
            f"plan_dtime ge '{start_date}' and "
            f"plan_dtime le '{end_date_plus_1}'"
        ),
        "$first": 10000
    }
    response = requests.get(url, params=params)

    print(response.status_code)
    print(response.url)

    response.raise_for_status()

    data = response.json()

    if "value" in data:
        return pd.DataFrame(data["value"])

    return pd.DataFrame(data)

In [ ]:
forecast_start = pd.Timestamp(f"{FORECAST_DATE} 00:00:00")
forecast_end = pd.Timestamp(f"{FORECAST_DATE} 23:00:00")

history_start = forecast_start - pd.Timedelta(hours=240)

full_index = pd.date_range(
    start=history_start,
    end=forecast_end,
    freq="h"
)

base = pd.DataFrame({"timestamp": full_index})
base.head(), base.tail()

In [ ]:
pse_start = (forecast_start - pd.Timedelta(hours=240)).strftime("%Y-%m-%d")
pse_end = forecast_end.strftime("%Y-%m-%d")

In [ ]:
load_raw = download_pse_pk5l_wp(
    start_date=pse_start,
    end_date=pse_end
)
load_raw.head(), load_raw.tail()

In [ ]:
pse_forecast = load_raw.copy()

# Dostosuj nazwy po sprawdzeniu load_raw.columns
pse_forecast = pse_forecast.rename(columns={
    "plan_dtime": "timestamp",
    "grid_demand_fcst": "load",
    "fcst_pv_tot_gen": "pv_gen",
    "fcst_wi_tot_gen": "fw_gen"
})

pse_forecast["timestamp_raw"] = pse_forecast["timestamp"].astype(str)

pse_forecast["timestamp"] = (
    pse_forecast["timestamp_raw"]
    .str.replace("03a", "03", regex=False)
)

pse_forecast["timestamp"] = pd.to_datetime(
    pse_forecast["timestamp"],
    errors="coerce"
)

for col in ["load", "pv_gen", "fw_gen"]:
    pse_forecast[col] = pd.to_numeric(pse_forecast[col], errors="coerce")

pse_forecast = (
    pse_forecast
    .dropna(subset=["timestamp", "load", "pv_gen", "fw_gen"])
    .groupby("timestamp", as_index=False)[["load", "pv_gen", "fw_gen"]]
    .mean()
    .sort_values("timestamp")
    .reset_index(drop=True)
)
pse_forecast = pse_forecast[["timestamp", "load", "pv_gen", "fw_gen"]]

In [ ]:
pse_forecast.describe()

In [ ]:
base = base.merge(
    spot_hist[["timestamp", "price_spot"]],
    on="timestamp",
    how="left"
)

base = base.merge(
    pse_forecast[["timestamp", "load", "pv_gen", "fw_gen"]],
    on="timestamp",
    how="left"
)

base.head()

In [ ]:
cols_to_interpolate = ["load", "pv_gen", "fw_gen"]

base = base.sort_values("timestamp").copy()

base[cols_to_interpolate] = (
    base[cols_to_interpolate]
    .interpolate(method="linear", limit_direction="both")
)

print(base[cols_to_interpolate].isna().sum())

In [ ]:
print(base.isna().sum())

In [ ]:
def download_open_meteo_weather(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "wind_speed_10m",
            "cloud_cover"
        ],
        "timezone": "Europe/Warsaw"
    }
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    
    data = response.json()["hourly"]
    weather = pd.DataFrame(data)
    weather["timestamp"] = pd.to_datetime(weather["time"])
    weather = weather.drop(columns=["time"])
    
    return weather

In [ ]:
weather = download_open_meteo_weather(
    LAT,
    LON,
    base["timestamp"].min().strftime("%Y-%m-%d"),
    base["timestamp"].max().strftime("%Y-%m-%d")
)

base = base.merge(weather, on="timestamp", how="left")

base.head()

In [ ]:
df_pred = base.copy()

df_pred["year"] = df_pred["timestamp"].dt.year
df_pred["month"] = df_pred["timestamp"].dt.month
df_pred["day"] = df_pred["timestamp"].dt.day
df_pred["hour"] = df_pred["timestamp"].dt.hour
df_pred["dayofweek"] = df_pred["timestamp"].dt.dayofweek
df_pred["dayofyear"] = df_pred["timestamp"].dt.dayofyear
df_pred["is_weekend"] = df_pred["dayofweek"].isin([5, 6]).astype(int)

df_pred["hour_sin"] = np.sin(2 * np.pi * df_pred["hour"] / 24)
df_pred["hour_cos"] = np.cos(2 * np.pi * df_pred["hour"] / 24)

df_pred["month_sin"] = np.sin(2 * np.pi * df_pred["month"] / 12)
df_pred["month_cos"] = np.cos(2 * np.pi * df_pred["month"] / 12)

df_pred["dayofyear_sin"] = np.sin(2 * np.pi * df_pred["dayofyear"] / 365)
df_pred["dayofyear_cos"] = np.cos(2 * np.pi * df_pred["dayofyear"] / 365)

In [ ]:
df_pred["res_gen"] = df_pred["fw_gen"] + df_pred["pv_gen"]
df_pred["residual_load"] = df_pred["load"] - df_pred["res_gen"]

df_pred["res_share"] = df_pred["res_gen"] / df_pred["load"]
df_pred["pv_share"] = df_pred["pv_gen"] / df_pred["load"]
df_pred["fw_share"] = df_pred["fw_gen"] / df_pred["load"]

In [ ]:
BASE_TEMP_HEATING = 18
BASE_TEMP_COOLING = 22

df_pred["heating_degree"] = np.maximum(0, BASE_TEMP_HEATING - df_pred["temperature_2m"])
df_pred["cooling_degree"] = np.maximum(0, df_pred["temperature_2m"] - BASE_TEMP_COOLING)

In [ ]:
pl_holidays = holidays.Poland(years=sorted(df_pred["year"].unique()))

df_pred["date"] = df_pred["timestamp"].dt.date
df_pred["is_holiday"] = df_pred["date"].isin(pl_holidays).astype(int)

df_pred["is_weekend_or_holiday"] = (
    (df_pred["is_weekend"] == 1) |
    (df_pred["is_holiday"] == 1)
).astype(int)

In [ ]:
price_lags = [24, 48, 72, 168]

for lag in price_lags:
    df_pred[f"price_lag_{lag}h"] = df_pred["price_spot"].shift(lag)

In [ ]:
system_lags = [24, 48, 72, 168]

for lag in system_lags:
    df_pred[f"load_lag_{lag}h"] = df_pred["load"].shift(lag)
    df_pred[f"pv_lag_{lag}h"] = df_pred["pv_gen"].shift(lag)
    df_pred[f"fw_lag_{lag}h"] = df_pred["fw_gen"].shift(lag)
    df_pred[f"residual_load_lag_{lag}h"] = df_pred["residual_load"].shift(lag)

In [ ]:
rolling_windows = [24, 48, 72, 168]

for window in rolling_windows:
    df_pred[f"price_roll_mean_{window}h"] = (
        df_pred["price_spot"].shift(24).rolling(window).mean()
    )
    
    df_pred[f"price_roll_std_{window}h"] = (
        df_pred["price_spot"].shift(24).rolling(window).std()
    )
    
    df_pred[f"residual_load_roll_mean_{window}h"] = (
        df_pred["residual_load"].shift(24).rolling(window).mean()
    )
    
    df_pred[f"load_roll_mean_{window}h"] = (
        df_pred["load"].shift(24).rolling(window).mean()
    )

In [ ]:
forecast_rows = df_pred[
    (df_pred["timestamp"] >= forecast_start) &
    (df_pred["timestamp"] <= forecast_end)
].copy()

missing_features = [col for col in features if col not in forecast_rows.columns]

if missing_features:
    raise ValueError(f"Brakuje cech wymaganych przez model: {missing_features}")

X_forecast = forecast_rows[features].copy()

print("Braki w X_forecast:")
print(X_forecast.isna().sum()[X_forecast.isna().sum() > 0])

In [ ]:
forecast_rows["price_spot_forecast"] = model.predict(X_forecast)

spot_forecast_d1 = forecast_rows[[
    "timestamp",
    "price_spot_forecast",
    "load",
    "pv_gen",
    "fw_gen",
    "residual_load",
    "temperature_2m"
]].copy()

spot_forecast_d1

In [ ]:
spot_forecast_d1.to_excel(
    f"prognoza_SPOT_{FORECAST_DATE}.xlsx",
    index=False
)
print(f"Zapisano prognozę dla dnia {FORECAST_DATE}")